# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² Kilifi County Mental Health Survey dataset using the `mlcroissant` library. All references to record sets, fields, and data columns use their `@id` identifiers for clarity and reproducibility.

### Dataset Source
This data is sourced from a Croissant schema URL, ensuring standardized structure and metadata accessibility.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access and print high-level metadata
metadata = dataset.metadata
print(f"{metadata.name}\n{metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"Cite as: {getattr(metadata, 'citeAs', None)}")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview
Review available record sets, their fields, and explore their `@id`s.

**Note:** All references to Croissant entities (record sets, fields, columns) are given by their `@id`.

In [ ]:
# Show all record sets and their IDs
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the schema. Check the dataset metadata for details.")
else:
    print("Record sets in the dataset:\n")
    for rs in record_sets:
        print(f"- Name: {getattr(rs, 'name', None)}\n  @id: {rs.id}\n  Description: {getattr(rs, 'description', '')}\n")
        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields (@id):")
            for f in rs.fields:
                print(f"    - {f.id} ({getattr(f, 'name', '')})")
            print()

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame. Reference both the record set and field(s) by their `@id`.

> **Tip:** If exploring a specific variable/field, use its `@id` as shown above.

In [ ]:
# Gather all record set IDs
rs_ids = [rs.id for rs in dataset.record_sets]
print(f"All available record set @id's: {rs_ids}\n")

# Load records from each record set by @id
dataframes = {}
for rs in dataset.record_sets:
    records = list(dataset.records(record_set=rs.id))
    dataframes[rs.id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from record set '{rs.name}' (@id: {rs.id})")

# Select the main record set for demonstration (e.g., the first one if unsure)
if rs_ids:
    main_rs_id = rs_ids[0]
    print(f"\nColumns in record set '{main_rs_id}':\n", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No record sets found. Cannot proceed to extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply core EDA operations such as filtering, normalization, and grouping. All fields referenced by their `@id`s.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# -----
# Replace these IDs by inspecting section 2/3 outputs (see .columns)
# Example choices for fields: choose GAD-7, PHQ-9 score column, or a demographic field
# Here we try to auto-detect a numeric score field (commonly present in public health surveys)
# Otherwise, use your field-of-interest @id directly
main_df = dataframes[main_rs_id]
numeric_candidate_ids = [col for col in main_df.columns if any(s in col.lower() for s in ['score', 'gad7', 'phq9', 'psq', 'age'])]
if numeric_candidate_ids:
    numeric_field_id = numeric_candidate_ids[0]
else:
    # fallback: first numeric-looking column
    numeric_field_id = None
    for col in main_df.columns:
        if np.issubdtype(main_df[col].dtype, np.number):
            numeric_field_id = col
            break
if not numeric_field_id:
    raise ValueError("No numeric field found for EDA. Adjust field selection above.")

# Filter records where numeric field > threshold (e.g., 10)
threshold = 10
filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce') > threshold].copy()
print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized '{numeric_field_id}' for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a demographic field if present (e.g., 'age_group', 'gender', etc.)
group_candidate_ids = [col for col in main_df.columns if any(s in col.lower() for s in ['gender', 'sex', 'group', 'marital', 'education'])]
if group_candidate_ids:
    group_field_id = group_candidate_ids[0]
    print(f"\nGrouping by field: {group_field_id}\n")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping analysis.")

## 5. Visualization
Visualize distributions and relationships between variables, referencing fields by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main numeric field
plt.figure(figsize=(8, 5))
sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of '{numeric_field_id}'")
plt.xlabel(numeric_field_id)
plt.show()

# Box plot by group (if available)
if group_candidate_ids:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to use the `mlcroissant` library to load, inspect, and process the FAIR² Kilifi County Mental Health Survey dataset. 

- **Data access and exploration** is seamlessly structured using the Croissant schema and entity `@id` references.
- **EDA** and **visualizations** highlight how survey scores distribute and how they might be associated with demographic variables.

You can use the same techniques for further, domain-specific analysis or data wrangling on this and other Croissant-structured datasets.